# FoldPipe: Baseline Training on Kaggle

This notebook establishes the baseline performance for Molecular Dynamics simulation using `TorchMD-Net` on the MD17 dataset.

In [ ]:
!pip install torchmd-net

In [ ]:
import torch
from torchmdnet.datasets import MD17
from torch_geometric.loader import DataLoader
import time

# PyTorch 2.6 changed torch.load default to weights_only=True, which breaks torchmd-net dataset caching.
# We monkey-patch torch.load to default to False for our trusted dataset.
_original_load = torch.load
torch.load = lambda *args, **kwargs: _original_load(*args, **{**kwargs, 'weights_only': False})

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Load MD17 dataset (Aspirin by default for testing)
dataset = MD17('./md17_data', molecules='aspirin')
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f"Loaded dataset with {len(dataset)} samples.")

In [ ]:
# Baseline Profiling Loop
# Here we simulate training by iterating through the dataloader and monitoring bottlenecks.
start_time = time.time()
for i, batch in enumerate(dataloader):
    # Move batch to device
    z = batch.z.to(device)
    pos = batch.pos.to(device)
    y = batch.y.to(device)
    
    if i % 100 == 0:
        print(f"Batch {i} processed in {time.time() - start_time:.2f} seconds")
        start_time = time.time()
    
    if i > 500:
        break # Just profile a subset for now
